# Honeypot Documentation Audit

**CSCI 5996 - Project 1**  
Muhammad Wahab Khan

An agentic pipeline that audits open-source honeypot repositories on GitHub and reports whether their documentation addresses the ethical or legal considerations of deploying one.

## Why this is not a simulated agent

The examples in Chapters 1–5 use placeholder handlers - `booking_handler(req)` returns the string `"Booking Handler processed '{req}'"` and nothing is booked. This pipeline calls the live GitHub REST API. Every repository it reports on is fetched at run time, and every quoted sentence is checked against the text that was actually retrieved. Nothing is stubbed.

The question it answers is a real one from my research: of the tools that present themselves as honeypots, how many tell an operator anything about the ethics or legality of running one?

## Patterns

| Pattern | Chapter | Where | Role |
|---|---|---|---|
| Tool Use | 5 | `tools.py` | Three GitHub API tools: metadata, README, doc-file listing |
| Routing | 2 | `classify()` | honeypot / detection_tool / other - only honeypots continue |
| Prompt Chaining | 1 | `extract_ethics()` | Prose findings → structured JSON |
| Reflection | 4 | `verify()` | Per-quote verification against source; paraphrases dropped |
| Parallelization | 3 | `audit_all()` | Repositories audited concurrently, semaphore-bounded |

## Setup

Requires `.env` in this folder containing `GOOGLE_API_KEY`. See README.md.

In [39]:
import json
import time

import cache
import pipeline
import tools

print(f"Model: {pipeline.MODEL}")
print(f"Cache: {cache.stats()}")

Model: gemini-3.6-flash
Cache: {'github': 11, 'llm': 0}


---
# Pattern 1 - Tool Use (Chapter 5)

Three tools wrap the GitHub REST API. They are declared with LangChain's `@tool` decorator, so the same functions can be handed to a tool-calling agent, but the pipeline invokes them directly because the call order here is fixed and known.

The output below is live data, not a fixture.

In [40]:
for t in tools.ALL_TOOLS:
    print(f"{t.name}: {t.description.splitlines()[0]}")

fetch_repo_metadata: Fetch a GitHub repository's description, topics, stars, and language.
fetch_readme: Fetch the full README text of a GitHub repository.
list_doc_files: List documentation-like files at the root of a repository.


In [41]:
print(tools.get_metadata("cowrie/cowrie"))

Repository: cowrie/cowrie
Description: Cowrie SSH/Telnet Honeypot https://docs.cowrie.org/
Topics: attacker, cowrie, cowrie-ssh, deception, decoy, honeypot, kippo, scp, security, sftp, ssh, telnet, telnet-honeypot, threat-analysis, threat-sharing, threatintel
Language: Python
Stars: 6534
Archived: False


In [42]:
readme = tools.get_readme("cowrie/cowrie")
print(f"README length: {len(readme)} characters")
print(readme[:400], "...")

README length: 5725 characters
.. SPDX-FileCopyrightText: 2014 Upi Tamminen <desaster@gmail.com>
.. SPDX-FileCopyrightText: 2014-2025 Michel Oosterhof <michel@oosterhof.net>
..
.. SPDX-License-Identifier: BSD-3-Clause

Cowrie
######

What is Cowrie
*****************************************

Cowrie is a medium to high interaction SSH and Telnet honeypot
designed to log brute force attacks and the shell interaction
performed by t ...


---
# Pattern 2 - Routing (Chapter 2)

The awesome-honeypots list mixes actual honeypots with detection tools, log analysers, and data-visualisation dashboards. Auditing a log parser for honeypot-deployment ethics is meaningless, so classification happens first and only honeypots continue to extraction.

As the Routing chapter recommends, an unrecognised label falls through to a default branch rather than raising - the router can never crash the run.

In [43]:
candidates = [
    "cowrie/cowrie",              # SSH/Telnet honeypot
    "paralax/awesome-honeypots",  # a link list, not a tool
]

for repo in candidates:
    meta = tools.get_metadata(repo)
    rdme = tools.get_readme(repo)
    print(f"{repo:32} -> {pipeline.classify(repo, meta, rdme)}")

cowrie/cowrie                    -> honeypot
paralax/awesome-honeypots        -> other


---
# Pattern 3 - Prompt Chaining (Chapter 1)

Two steps rather than one. The first reads the documentation and reports ethics-related content as prose, quoting verbatim. The second converts that prose into JSON.

Splitting these matters. A single prompt asked to both *find* and *format* tends to produce nested objects or paraphrase while formatting. Separating the jobs means each prompt has one instruction to follow, and when the output is wrong it is obvious which step to fix - the debugging benefit the chapter claims for chaining.

In [44]:
REPO = "telekom-security/tpotce"

docs = "\n\n".join([
    tools.get_metadata(REPO),
    f"Documentation files: {tools.get_doc_files(REPO)}",
    tools.get_readme(REPO),
])

# Step 1 - prose findings
findings = pipeline.ask("summarise", pipeline.summarise_prompt.invoke({"docs": docs[:12000]}))
print(findings)

The documentation outlines several ethical, legal, privacy, and responsible-use considerations under its **Disclaimer** section:

* **User Responsibility and System Risk:** The documentation emphasizes that deployment and operational risks rest with the user, warning that system compromise is a possibility.
  > "You install and run T-Pot within your responsibility. Choose your deployment wisely as a system compromise can never be ruled out."

* **Responsible Disclosure:** Users are requested to report software issues responsibly.
  > "Report responsibly."

* **Sensitive Data and Privacy:** The documentation warns against storing sensitive data on the honeypot system.
  > "Honeypots - by design - should not host any sensitive data. Make sure you don't add any."

* **Data Sharing and Telemetry:** It details default data submission settings and how to opt out for privacy considerations.
  > "By default, your data is submitted to Sicherheitstacho. You can disable this in the config (`~/tpo

In [45]:
# Step 2 - structured JSON from those findings
extracted = pipeline.extract_ethics(docs)
print(json.dumps(extracted, indent=2))

{
  "has_ethics_statement": true,
  "quotes": [
    "You install and run T-Pot within your responsibility. Choose your deployment wisely as a system compromise can never be ruled out.",
    "Report responsibly.",
    "Honeypots - by design - should not host any sensitive data. Make sure you don't add any.",
    "By default, your data is submitted to Sicherheitstacho. You can disable this in the config (`~/tpotce/docker-compose.yml`) by removing the community data submission section."
  ],
  "summary": "The documentation outlines user operational risks, responsible disclosure expectations, privacy guidelines regarding sensitive data, and instructions for opting out of default telemetry data sharing."
}


---
# Pattern 4 - Reflection (Chapter 4)

A quote is only useful if it is real. The critic checks every extracted quote against the source text before it reaches the results.

**Design decision.** The chapter presents an LLM critic and notes separately that a deterministic check inside the loop - tests, validators - is the strongest form of critique. Here the two were compared directly. Normalising whitespace and testing for literal containment answers the question exactly, cannot itself hallucinate, and costs no quota. An LLM critic was implemented (`critic_prompt`, retained in `pipeline.py`) but adds uncertainty to a question that already has a certain answer, so the deterministic check is what ships.

**Verification is per quote, not per extraction.** The first version rejected the whole extraction if any quote failed. On T-Pot that discarded three verbatim quotes because a fourth was paraphrased, and the repository was wrongly recorded as documenting nothing. Checking each quote independently keeps the sound ones and drops only the invented one.

In [46]:
quotes = extracted.get("quotes", [])
haystack = " ".join(docs.split()).lower()

for q in quotes:
    ok = " ".join(q.split()).lower() in haystack
    print(f"[{'KEEP' if ok else 'DROP'}] {q[:90]}")

verdict, kept = pipeline.verify(docs, quotes)
print(f"\nVerdict: {verdict}")
print(f"Quotes retained: {len(kept)} of {len(quotes)}")

[KEEP] You install and run T-Pot within your responsibility. Choose your deployment wisely as a s
[KEEP] Report responsibly.
[KEEP] Honeypots - by design - should not host any sensitive data. Make sure you don't add any.
[DROP] By default, your data is submitted to Sicherheitstacho. You can disable this in the config

Verdict: PARTIAL (3/4 verified)
Quotes retained: 3 of 4


The dropped quote is a paraphrase of a real passage - the model summarised the telemetry opt-out instructions instead of copying them. Plausible, accurate in substance, and not a quotation. In a literature review where each claim must be traceable to a source, that distinction is the whole point of the check.

---
# Pattern 5 - Parallelization (Chapter 3)

Repositories are independent of one another - a textbook fan-out. Each audit is network-bound rather than CPU-bound, so `asyncio.to_thread` runs them concurrently and `asyncio.gather` collects the results.

A semaphore bounds concurrency. Without it, fanning out over the full input set would exceed both GitHub's unauthenticated rate limit and the model's per-minute quota - the design complexity the chapter lists as parallelization's cost.

The timing below is measured with a warm cache, so it shows the orchestration overhead rather than the network saving. Clear the cache with `cache.clear()` to see the real difference.

In [47]:
REPOS = [
    "cowrie/cowrie",
    "telekom-security/tpotce",
    "thinkst/opencanary",
]

start = time.perf_counter()
serial = [pipeline.audit_one(r) for r in REPOS]
serial_time = time.perf_counter() - start

start = time.perf_counter()
concurrent = await pipeline.audit_all(REPOS, concurrency=3)
concurrent_time = time.perf_counter() - start

print(f"Serial:     {serial_time:.2f}s")
print(f"Concurrent: {concurrent_time:.2f}s")

Serial:     0.05s
Concurrent: 0.02s


---
# Full run

All five patterns over the input set.

**Quota note.** Each honeypot costs roughly three model calls. The Gemini free tier allows a limited number per day, so a large input set may need to be run across more than one session. Everything is cached on disk, so a re-run after a partial failure only pays for repositories not yet seen.

In [ ]:
INPUT_SET = [
    # already audited
    "cowrie/cowrie",                    # SSH/Telnet honeypot
    "telekom-security/tpotce",          # multi-honeypot platform
    "thinkst/opencanary",               # network decoy daemon

    # additional honeypots
    "DinoTools/dionaea",                # malware-capture honeypot
    "mushorg/conpot",                   # ICS/SCADA honeypot
    "mushorg/snare",                    # web application honeypot
    "johnnykv/heralding",               # credential-capturing honeypot
    "foospidy/HoneyPy",                 # low-interaction honeypot
    "honeytrap/honeytrap",              # extensible honeypot framework

    # should NOT classify as honeypot - tests the router
    "paralax/awesome-honeypots",        # a link list
    "ThreatConnect-Inc/tcex",           # threat intel SDK
    "MISP/MISP",                        # threat sharing platform
]

results = await pipeline.audit_all(INPUT_SET, concurrency=1)
print(pipeline.to_table(results))

d:\Ph.D. CS\Ph.D. CS Coursework\Fall 26 Special Topics - Agentic AI\Projects\Project 1\.venv\Lib\site-packages\debugpy\_vendored\pydevd\pydevd_file_utils.py:307: RuntimeWarning: coroutine 'audit_all' was never awaited
  def _normcase_windows(filename):


GoogleRateLimitError: Error calling model 'gemini-3.6-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-3.6-flash\nPlease retry in 3.245220121s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 

In [49]:
print(pipeline.summary(results))

3 repositories audited. 3 classified as honeypots. 1 of those (33%) document ethical or legal considerations.


In [50]:
for r in results:
    if r.ethics_quotes:
        print(f"\n### {r.repo}  [{r.verification}]")
        for q in r.ethics_quotes:
            print(f"  - {q}")


### telekom-security/tpotce  [PARTIAL (3/4 verified)]
  - You install and run T-Pot within your responsibility. Choose your deployment wisely as a system compromise can never be ruled out.
  - Report responsibly.
  - Honeypots - by design - should not host any sensitive data. Make sure you don't add any.


In [51]:
with open("results.json", "w", encoding="utf-8") as f:
    json.dump([r.to_dict() for r in results], f, indent=2)

print(f"Wrote results.json - {len(results)} repositories")
print(f"Cache: {cache.stats()}")

Wrote results.json - 3 repositories
Cache: {'github': 11, 'llm': 0}


---
# Findings and limitations

**What the audit measures.** Only what the repository itself publishes: metadata, README, and root-level documentation files. Cowrie's full documentation lives at docs.cowrie.org, off-repo and outside this scope. So the claim is that *the repository documentation* does not address ethics, not that the project never does. Extending the tools to follow documentation links would widen the scope and is the obvious next step.

**On the classifier.** Routing decisions come from an LLM reading a README, and no ground-truth labels exist for this corpus. A hand-labelled sample would be needed to state an accuracy figure. The Routing chapter's suggestion - a rule-based triage layer in front of the LLM router - would also cut cost, since repositories with `honeypot` in their topics need no model call at all.

**What reflection bought.** One repository in the first three had a fabricated quote in an otherwise accurate extraction. Without per-quote verification that paraphrase would have entered the results as a citation. The rate is too small a sample to generalise from, but it is not zero, and in a literature review a single invented quote is a serious error.